# 05 — Streaming Tool Calls: Examples

> Accumulate partial JSON arguments across chunks and execute parallel tool calls.

**What you'll build:**
1. Visualize raw tool call chunks
2. Correct accumulator implementation
3. Parallel tool call handling
4. JSON error recovery

In [ ]:
import os, json
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint
from rich.table import Table
from rich.console import Console

load_dotenv()
client = OpenAI()
console = Console()

TOOLS = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get weather for a city",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
    }},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Evaluate a math expression",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}
    }},
]
print('✅ Setup complete')

---
## Part 1: Visualize Raw Tool Call Chunks

In [ ]:
# Show exactly what arrives in each chunk when a tool call is made
print("🔍 Inspecting raw tool call chunks...\n")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
    tools=TOOLS, stream=True
)

tc_chunks = []
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.tool_calls or chunk.choices[0].finish_reason == "tool_calls":
        tc_chunks.append(chunk)

table = Table(title="Tool Call Raw Chunks", show_header=True)
table.add_column("#", style="cyan", width=4)
table.add_column("tc.index", width=8)
table.add_column("tc.id", style="yellow", width=14)
table.add_column("tc.function.name", style="green", width=18)
table.add_column("tc.function.arguments", style="white", width=30)
table.add_column("finish_reason", style="red", width=14)

for i, chunk in enumerate(tc_chunks):
    delta = chunk.choices[0].delta
    fr = str(chunk.choices[0].finish_reason)
    if delta.tool_calls:
        for tc in delta.tool_calls:
            table.add_row(
                str(i), str(tc.index),
                repr(tc.id) if tc.id else "None",
                repr(tc.function.name) if tc.function and tc.function.name else "None",
                repr(tc.function.arguments) if tc.function else "None",
                fr
            )
    else:
        table.add_row(str(i), "-", "-", "-", "-", fr)

console.print(table)

---
## Part 2: Tool Call Accumulator

In [ ]:
def stream_and_collect_tool_calls(messages: list[dict]) -> dict:
    """Stream and correctly accumulate all tool call arguments."""
    full_text = ""
    tool_calls_acc = {}
    finish_reason = None

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS, stream=True
    )

    for chunk in stream:
        delta = chunk.choices[0].delta
        finish_reason = chunk.choices[0].finish_reason or finish_reason

        if delta.content:
            full_text += delta.content

        if delta.tool_calls:
            for tc in delta.tool_calls:
                idx = tc.index
                if idx not in tool_calls_acc:
                    tool_calls_acc[idx] = {"id": "", "name": "", "arguments": ""}
                if tc.id:                       tool_calls_acc[idx]["id"]        += tc.id
                if tc.function and tc.function.name:       tool_calls_acc[idx]["name"]      += tc.function.name
                if tc.function and tc.function.arguments:  tool_calls_acc[idx]["arguments"] += tc.function.arguments

    # Parse after full accumulation
    parsed = {}
    for idx, tc in tool_calls_acc.items():
        try:
            args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            error = None
        except json.JSONDecodeError as e:
            args = {}
            error = str(e)
        parsed[idx] = {**tc, "args": args, "parse_error": error}

    return {"text": full_text, "finish_reason": finish_reason, "tool_calls": parsed}


# Test
result = stream_and_collect_tool_calls([
    {"role": "user", "content": "What is the weather in London, and what is 42 * 8?"}
])

rprint(f"\n[bold]finish_reason:[/bold] {result['finish_reason']}")
rprint(f"[bold]Tool calls detected:[/bold] {len(result['tool_calls'])}")
for idx, tc in result['tool_calls'].items():
    rprint(f"  [{idx}] [green]{tc['name']}[/green]: id={tc['id'][:15]}...")
    rprint(f"       raw args:    {repr(tc['arguments'])}")
    rprint(f"       parsed args: {tc['args']}")

---
## Part 3: Execute Both Tool Calls

In [ ]:
# Mock executors
def get_weather(city): return f"{city}: 18°C, cloudy"
def calculate(expression):
    try: return str(eval(expression, {"__builtins__": {}}, {}))
    except: return "error"

EXECUTOR = {"get_weather": get_weather, "calculate": calculate}

messages = [{"role": "user", "content": "What is the weather in London, and what is 42 * 8?"}]

# Step 1: Stream and collect tool calls
r = stream_and_collect_tool_calls(messages)

# Step 2: Build assistant message (with raw argument strings)
tc_list_for_msg = [{
    "id": tc["id"], "type": "function",
    "function": {"name": tc["name"], "arguments": tc["arguments"]}
} for tc in r['tool_calls'].values()]

messages.append({"role": "assistant", "content": r['text'] or None, "tool_calls": tc_list_for_msg})

# Step 3: Execute tools and add results
rprint("\n[bold]Executing tools:[/bold]")
for tc in r['tool_calls'].values():
    fn = EXECUTOR.get(tc['name'])
    result = fn(**tc['args']) if fn else "tool not found"
    rprint(f"  🔧 {tc['name']}({tc['args']}) → [green]{result}[/green]")
    messages.append({"role": "tool", "tool_call_id": tc['id'], "content": str(result)})

# Step 4: Stream final answer
rprint("\n[bold cyan]🤖 Final answer:[/bold cyan]", end=" ")
stream2 = client.chat.completions.create(model="gpt-4o-mini", messages=messages, stream=True)
for chunk in stream2:
    c = chunk.choices[0].delta.content or ""
    print(c, end="", flush=True)
print()

---
## Summary

| Key Point | Detail |
|---|---|
| JSON arrives in pieces | Must concatenate `arguments` strings across all chunks |
| Use `tc_delta.index` | Tracks which tool call each chunk belongs to |
| Parse AFTER stream ends | Never call `json.loads()` on partial arguments |
| Keep raw string in history | Use `tc['arguments']` (str) not parsed dict in messages |

**Next:** `06_async_streaming` — concurrent streaming with asyncio.